In [6]:
### 중요: 이 코드는 인과적 어텐션 메커니즘을 구현한 PyTorch 모듈입니다.
import torch
import torch.nn as nn

class CausalAttention(nn.Module): #하나의 모듈로 선언
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        """
        Args:
            d_in: 입력 벡터의 차원 크기
            d_out: 출력(및 쿼리/키/밸류) 벡터의 차원 크기
            context_length: 모델이 한 번에 처리할 수 있는 최대 문맥 길이 (토큰 수)
            dropout: 드롭아웃 확률
            qkv_bias: 선형 레이어에 편향(bias)을 사용할지 여부
        """
        super().__init__()

        #중요
        # 1. 쿼리(Query), 키(Key), 밸류(Value)를 만들기 위한 선형 투영 레이어 정의
        # 입력 벡터(d_in)를 각각의 목적에 맞는 벡터(d_out)로 변환합니다.
        # TODO: Q/K/V 투영 레이어 타입과 인자를 채우세요.
        # 힌트: 입력 d_in, 출력 d_out을 사용하는 Linear 레이어를 선언하세요.
        #self.W_query = nn.????(????, d????, bias=qkv_bias)
        #self.W_key   = nn.????(????, ????, bias=qkv_bias)
        #self.W_value = nn.????(????, ????, bias=qkv_bias)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # self.W_query는 객체가 저장된다. 
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 2. 과적합(Overfitting) 방지를 위한 드롭아웃 설정
        # 학습 데이터에 너무과하게 맞춰지는 과적합을 방지하기 위해 뉴런의 일부를 무작위로 꺼버림
        self.dropout = nn.Dropout(dropout)

        #중요
        # 3. 인과적 마스크(Causal Mask) 생성 및 버퍼 등록
        # 'register_buffer'를 사용하면 역전파(학습) 대상은 아니지만, 모델의 상태(state_dict)로 저장됩니다.
        # torch.triu(..., diagonal=1): 대각선(0) 위쪽 삼각형 부분만 1로 채웁니다.
        # 즉, '미래의 정보' 위치에 1을 표시하여 나중에 가릴(masking) 준비를 합니다.
        self.register_buffer(
            'mask', 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    # 중요
    def forward(self, x): # 실질적인 연산이 일어남
        # x.shape: [배치 크기(b), 토큰 개수(num_tokens), 입력 차원(d_in)]
        b, num_tokens, d_in = x.shape 

        # 입력 x를 통과시켜 현재 시점의 관심사(Query), 검색 대상(Key), 정보 내용(Value)을 추출합니다.
        # region [Q, K, V 벡터 계산]
        # TODO: Q, K, V 계산 호출 대상을 채우세요.
        # 힌트: W_query, W_key, W_value를 각각 x에 적용하면 됩니다.
        #keys = self.????(????)        # Shape: [b, num_tokens, d_out]
        #queries = self.?????(????)   # Shape: [b, num_tokens, d_out]
        #values = self.?????(????)    # Shape: [b, num_tokens, d_out]
        keys = self.W_key(x)        # Shape: [b, num_tokens, d_out]
        queries = self.W_query(x)   # Shape: [b, num_tokens, d_out]
        values = self.W_value(x)    # Shape: [b, num_tokens, d_out]
        # endregion

        # Query와 Key의 내적(Dot Product)을 통해 각 토큰 간의 관련성을 구합니다.
        # keys.transpose(1, 2): 행렬 곱을 위해 차원을 뒤집습니다. (d_out 차원끼리 곱해짐)
        # region [어텐션 스코어(유사도) 계산]
        # TODO: 어텐션 스코어 행렬곱 피연산자를 채우세요.
        # 힌트: queries와 keys.transpose(1, 2)를 곱해 유사도 점수를 구합니다.
        # attn_scores = ???? @ ????.transpose(1, 2) #쿼리 키
        attn_scores = queries @ keys.transpose(1, 2) #쿼리 키
        #endregion

        # mask가 1인 위치(미래 시점의 토큰들)를 -무한대(-inf)로 채웁니다.
        # 이렇게 하면 나중에 Softmax를 거칠 때 확률이 0이 되어, 미래 정보를 참조하지 못하게 됩니다.
        # [:num_tokens, :num_tokens]: 입력 길이가 context_length보다 짧을 때를 대비해 크기를 맞춥니다.
        # region [인과적 마스킹 (Masking) - 미래 정보 차단]
        # TODO: 미래 시점의 어텐션 스코어를 가릴 값을 채우세요.
        # 힌트: Softmax를 통과하면 0이 되도록 마이너스 무한대(-torch.inf)를 입력하세요.
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],   # 대각선 위를 의미
            -torch.inf   # ????  #마이너스 무한대
        ) 
        # endregion

        # 스케일링(/ keys.shape[-1]**0.5): 차원이 커질수록 내적 값이 커져 기울기 소실이 오는 것을 방지합니다.
        # Softmax: 점수를 확률(0~1 사이, 합은 1)로 변환합니다.
        # region [어텐션 가중치(Weights) 계산 및 스케일링]
        # TODO: 어텐션 가중치 함수명을 채우세요.
        # 힌트: 스케일 적용 후 softmax를 사용해 확률로 변환하세요.
        #attn_weights = torch.????(
        attn_weights = torch.softmax(  # 어텐션스코어를 확률로 바꿈
            attn_scores / keys.shape[-1]**0.5, dim=-1  #dk로 나누기. **0.5는 루트임
        )
        # endregion

        # 계산된 가중치 중 일부를 무작위로 0으로 만들어 모델이 특정 토큰에만 의존하는 것을 막습니다.
        # region [드롭아웃 적용]
        attn_weights = self.dropout(attn_weights)
        # endregion

        # 어텐션 가중치(확률)를 기반으로 Value(정보)들을 가중 합산합니다.
        # 결과적으로 "현재 토큰과 관련이 깊은 과거 토큰들의 정보"가 진하게 섞인 벡터가 됩니다.
        # region [문맥 벡터(Context Vector) 생성]
        # TODO: 컨텍스트 벡터 계산 피연산자를 채우세요.
        # 힌트: attn_weights와 values를 곱해 최종 컨텍스트 벡터를 얻습니다.
        # context_vec = attn_weights @ ????  # 밸류
        context_vec = attn_weights @ values
        # endregion
        
        return context_vec  #문맥벡터를 만듬


In [7]:
import torch

inputs = torch.tensor(   # 아랫 값들은 가정한 값들임
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in = inputs.shape[1]   # 입력 차원 (d=3)
d_out = 2                # Q, K, V의 출력 차원 (d=2)

batch = torch.stack((inputs, inputs), dim=0) #인풋2개를 배치로 만듬
# --- 실행 예시 ---
torch.manual_seed(123)

# 가정: batch 변수가 이미 정의되어 있다고 가정 (예: b=2, num_tokens=6, d_in=...)
# context_length는 모델이 허용하는 최대 길이이므로, 현재 배치의 길이와 같거나 더 길게 설정합니다.
context_length = batch.shape[1] 

print("context_length:", context_length)

# TODO: 어텐션 모듈 호출 대상을 채우세요.
# 힌트: 직전에 생성한 CausalAttention 인스턴스(ca)를 호출하면 됩니다.
ca = CausalAttention(d_in, d_out, context_length, 0.0)
#context_vecs = ????(batch)
context_vecs = ca(batch)  # forward 함수가 실행됨

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

context_length: 6
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [8]:
#중요
# 어텐션을 여러개 쌓아서 배열로 만듬
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) 
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1) # 계산한 결과를 합침. 계산이 별렬로 이뤄지지는 않음 (for문 돔)


torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


In [9]:
# QKV를 크게 하나 만들든 후, head_dim 만큼 잘라서 사용
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # 한번에 연산이 끝남. 멀티헤드어텐션
        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 

        # 자른 head 를 원복시킴. Projection
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
# d_out = 2
d_out = 4
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


<img src="llm_from_scratch/images/llm_from_scratch/ch03_compressed/26.webp" width="600px">

## Simulation
- https://poloclub.github.io/transformer-explainer/
- https://bbycroft.net/llm
